# Figure 3c

This notebook calculates the mean bigWig signal for each ChromHMM state and plots the heatmaps used for Figure 3c.


## Environment

The code was run with Python 3.9.12. Use the commands below to create a matching conda environment.

```bash
conda create -n chromhmm-signal python=3.9.12
conda activate chromhmm-signal
conda install -c conda-forge -c bioconda \
    numpy=1.24.4 \
    pandas=1.5.3 \
    "matplotlib-base=3.7.*" \
    seaborn=0.12.2 \
    pyBigWig=0.3.22
```

Library versions:

- numpy 1.24.4
- pandas 1.5.3
- matplotlib 3.7.3
- seaborn 0.12.2
- pyBigWig 0.3.22

`os`, `glob`, and `collections` are part of the Python standard library, so they do not need to be installed.


## Calculate the signal matrix

Load the 12 chromatin state BED files and calculate the length-weighted mean signal for each bigWig sample.


In [ ]:
import os, glob
from pathlib import Path
import pyBigWig
import pandas as pd
from collections import defaultdict

DATA_ROOT = Path(os.environ.get("GAO2026_DATA_ROOT", "data")).expanduser().resolve()
MODULE_ROOT = DATA_ROOT / "16_ChromHMM_Fig3c"
if not MODULE_ROOT.is_dir():
    MODULE_ROOT = DATA_ROOT / "revision" / "16_ChromHMM_Fig3c"
EXTERNAL_ROOT = Path(os.environ.get("GAO2026_EXTERNAL_ROOT", DATA_ROOT)).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("GAO2026_OUTPUT_ROOT", "outputs")).expanduser().resolve() / "16_ChromHMM_Fig3c"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
STATE_DIR = str(MODULE_ROOT / "state_beds")
BW_DIR = str(EXTERNAL_ROOT / "16_ChromHMM_Fig3c" / "bw")

SAMPLE_MAP = {
    "SRX3898502": "BRD4",
    "ERX2789783": "HDAC3",
    # ===== CHANGED: removed the line  "SRX2875257": "RNAPII"  =====
    # RNAPII is not shown in the figure (it was dropped just before plotting),
    # so under the new "mapped-only" logic below it is no longer processed.
    "SRX029864": "SIN3A",
    "SRX047138": "HDAC2",
    "SRX1091746": "NCOR2",
    "SRX1091748": "H3K27ac",
    "SRX2539820": "HDAC1",
    "SRX4403303": "CHD4",
    "SRX194535": "p300",
    "SRX6098709": "NCOR1",
    "SRX13298161": "ATAC-seq",
    "ENCFF806NDV": "H3K4me3",
    "ENCFF705OWT": "H3K9ac",
    "ENCFF519ZAT": "H3K9me3",
    "ENCFF182FTP": "H3K27me3",
    "SRX017056": "RNAPII-Ser5ph",
    "SRX017057": "RNAPII-Ser2ph",
    "SRX873340": "SIRT6",
}

# Load the 12 state BED files and group intervals by chromosome
states = {}  # state_name -> {chrom: [(start, end), ...]}
for bed in sorted(glob.glob(os.path.join(STATE_DIR, "*.bed"))):
    fname = os.path.basename(bed)
    # Example: 10_Transcription_Elongation.bed -> Transcription_Elongation
    state_name = "_".join(fname.split("_")[1:]).replace(".bed", "")
    chrom_dict = defaultdict(list)
    with open(bed) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            chrom, start, end = line.split()[:3]
            chrom_dict[chrom].append((int(start), int(end)))
    states[state_name] = chrom_dict

state_names = list(states.keys())

# Calculate the mean signal for each state in each bigWig file
result = pd.DataFrame(index=[], columns=state_names, dtype=float)

for bw_path in sorted(glob.glob(os.path.join(BW_DIR, "*.bw"))):
    sample_id = os.path.basename(bw_path).replace(".bw", "")
    # ===== CHANGED: process ONLY bigWigs that have an explicit SAMPLE_MAP entry.
    # Any file in the folder without a mapping (e.g. SRX12798486, mm10-blacklist,
    # mm10.chrom.sizes) is now skipped instead of being plotted under its raw ID.
    # Old behaviour was:
    #     if sample_id in ["mm10-blacklist", "mm10.chrom.sizes"]:
    #         continue
    #     factor_name = SAMPLE_MAP.get(sample_id, sample_id)
    if sample_id not in SAMPLE_MAP:
        continue
    factor_name = SAMPLE_MAP[sample_id]  # Map the sample ID to the factor name
    # ===== end of change =====
    print("Processing:", sample_id, "->", factor_name)

    bw = pyBigWig.open(bw_path)
    row_values = {}

    for state_name, chrom_dict in states.items():
        total_signal = 0.0
        total_bp = 0

        for chrom, intervals in chrom_dict.items():
            if chrom not in bw.chroms():
                continue
            for start, end in intervals:
                length = end - start
                if length <= 0:
                    continue
                # bw.stats returns a list; use the mean value from the first bin
                mean_val = bw.stats(chrom, start, end, type="mean", nBins=1)[0]
                if mean_val is None:
                    continue
                total_signal += mean_val * length
                total_bp += length

        if total_bp > 0:
            row_values[state_name] = total_signal / total_bp  # Length-weighted mean signal for this state
        else:
            row_values[state_name] = 0.0

    bw.close()
    result.loc[factor_name] = pd.Series(row_values)

# Save the signal matrix
out_tsv = OUTPUT_ROOT / "chromState_signal_matrix.tsv"
result.to_csv(out_tsv, sep="\t")
print("Saved matrix to", out_tsv)

## Plot Figure 3c

Combine ChromHMM emission probabilities with the signal matrix, draw the two heatmaps, and save the figure as a PDF.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

# Input and output paths
EMISSIONS_FILE = MODULE_ROOT / "emissions_12.txt"
computed_matrix = OUTPUT_ROOT / "chromState_signal_matrix.tsv"
SIGNAL_MATRIX_FILE = computed_matrix if computed_matrix.exists() else MODULE_ROOT / "chromState_signal_matrix.tsv"
OUT_PDF = OUTPUT_ROOT / "Fig.3c.pdf"

# Font settings
plt.rcParams["font.family"] = "Arial"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# Use the white seaborn style; font sizes are set explicitly below
sns.set(style="white")

# Font size settings
FS_TITLE = 16          # Subplot titles
FS_XTICK = 16          # Top x-axis tick labels
FS_YTICK = 16          # Left y-axis tick labels
FS_CBAR_LABEL = 16     # Colorbar labels
FS_CBAR_TICK = 16     # Colorbar tick labels

# Chromatin state names
state_name_map = {
    1: "Heterochromatin",
    2: "Intergenic_Region",
    3: "Weakly_Repressed_Polycomb",
    4: "Repressed_Polycomb",
    5: "Bivalent_Promoter",
    6: "Weak_Enhancer",
    7: "Strong_Enhancer",
    8: "Enhancer",
    9: "Transcription_Transition",
    10: "Transcription_Elongation",
    11: "Insulator",
    12: "Active_Promoter",
}

# Load the signal and emission matrices
df_bottom = pd.read_csv(SIGNAL_MATRIX_FILE, sep="\t", index_col=0)  # factor x state
df_bottom = df_bottom.drop(index=["RNAPII"], errors="ignore")
state_order = df_bottom.columns.tolist()

em_raw = pd.read_csv(EMISSIONS_FILE, sep="\t")
em_raw["state_name"] = em_raw["State (Emission order)"].map(state_name_map)
em_raw = em_raw.set_index("state_name").drop(columns=["State (Emission order)"])

# Keep the same state order in both matrices
states_common = [s for s in state_order if s in em_raw.index]
state_labels = [s.replace("_", " ") for s in states_common]

# Left heatmap: state x mark, with reversed column order
em_left = em_raw.loc[states_common].iloc[:, ::-1]
em_left.index = state_labels
marks_labels = em_left.columns.tolist()

# Right heatmap: state x factor, with reversed columns and column-wise z-scores
df_state_factor = df_bottom[states_common].T.iloc[:, ::-1]  # state x factor
df_right_norm = df_state_factor.sub(df_state_factor.mean(axis=0), axis=1)
df_right_norm = df_right_norm.div(df_right_norm.std(axis=0), axis=1).fillna(0)
df_right_norm.index = state_labels
factor_labels = df_right_norm.columns.tolist()

# Plot the two heatmaps
fig, (ax_left, ax_right) = plt.subplots(
    1, 2,
    figsize=(11, 7),
    sharey=True,
    gridspec_kw={"width_ratios": [1, 1.7]}
)

# Left heatmap
hm1 = sns.heatmap(
    em_left, ax=ax_left, cmap="Blues", cbar=True,
    cbar_kws={
        "orientation": "horizontal",
        "pad": 0.08,
        "fraction": 0.05,
        "label": "ChromHMM\nemission probabilities",
    }
)

# Left title
ax_left.set_title("Probability of mark/state", pad=70, fontsize=FS_TITLE)

# Left x-axis labels at the top
ax_left.set_xlabel("")
ax_left.xaxis.tick_top()
ax_left.tick_params(axis="x", labeltop=True, labelbottom=False)
ax_left.set_xticklabels(marks_labels, rotation=-90, ha="center", fontsize=FS_XTICK)

# Left y-axis labels
ax_left.tick_params(axis="y", rotation=0, labelsize=FS_YTICK)

# Left colorbar label and tick sizes
cbar1 = hm1.collections[0].colorbar
cbar1.ax.xaxis.label.set_ha("center")   # Center the label horizontally
# cbar1.ax.xaxis.label.set_va("center")   # Center the label vertically
cbar1.ax.xaxis.label.set_size(FS_CBAR_LABEL)            # Label size
cbar1.ax.tick_params(labelsize=FS_CBAR_TICK)            # Tick size
# Show one decimal place
cbar1.formatter = FormatStrFormatter('%.1f')
cbar1.update_ticks()

# Right heatmap
hm2 = sns.heatmap(
    df_right_norm, ax=ax_right, cmap="RdBu_r", center=0.0, cbar=True,
    cbar_kws={
        "orientation": "horizontal",
        "pad": 0.08,
        "fraction": 0.05,
        "label": "Z-score",
    }
)

# Right title
ax_right.set_title("Factor Enrichment", pad=40, fontsize=FS_TITLE)

# Right x-axis labels at the top
ax_right.set_xlabel("")
ax_right.set_ylabel("")
ax_right.xaxis.tick_top()
ax_right.tick_params(axis="x", labeltop=True, labelbottom=False)
ax_right.set_xticklabels(factor_labels, rotation=-90, ha="center", fontsize=FS_XTICK)

# Hide y-axis labels on the right heatmap
ax_right.tick_params(axis="y", left=False, labelleft=False)

# Right colorbar label and tick sizes
cbar2 = hm2.collections[0].colorbar
cbar2.ax.xaxis.label.set_size(FS_CBAR_LABEL)
cbar2.ax.tick_params(labelsize=FS_CBAR_TICK)
cbar2.formatter = FormatStrFormatter('%.1f')
cbar2.update_ticks()

fig.tight_layout()
fig.savefig(OUT_PDF, dpi=600, bbox_inches="tight")
plt.show()

print("Saved to:", OUT_PDF)